<a href="https://colab.research.google.com/github/harshisrai/Colab/blob/main/CUDA_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install CUDA C++ plugin for Colab:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpagwbz7y_".


In [2]:
# Detect selected GPU and its NVIDA architecture:
import subprocess
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader,nounits")
if "not found" in gpu_info.lower(): raise RuntimeError("Error: No GPU found. Please select a GPU runtime environment.")
gpu_name, compute_cap = map(str.strip, gpu_info.split(','))
gpu_arch = f"sm_{compute_cap.replace('.', '')}"

print(f"{'GPU Name':<15}: {gpu_name}")
print(f"{'Architecture':<15}: {gpu_arch}")

GPU Name       : Tesla T4
Architecture   : sm_75


In [3]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include <stdio.h>

__global__ void hello_kernel() {
    int blockId = blockIdx.x;
    int threadId = threadIdx.x;
    int globalId = threadId + blockId * blockDim.x;

    printf("Hello from block %d, thread %d (global thread %d)\n", blockId, threadId, globalId);
}

int main() {
    int numBlocks = 2;
    int threadsPerBlock = 4;

    hello_kernel<<<numBlocks, threadsPerBlock>>>();
    cudaDeviceSynchronize();

    return 0;
}

Hello from block 1, thread 0 (global thread 4)
Hello from block 1, thread 1 (global thread 5)
Hello from block 1, thread 2 (global thread 6)
Hello from block 1, thread 3 (global thread 7)
Hello from block 0, thread 0 (global thread 0)
Hello from block 0, thread 1 (global thread 1)
Hello from block 0, thread 2 (global thread 2)
Hello from block 0, thread 3 (global thread 3)



In [5]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include<stdio.h>
#include<stdlib.h>

void add(int *a, int *b, int *c, int N){
    for(int i=0;i<N;i++){
        c[i]=a[i]+b[i];
    }
}

int main(int argc, char** argv){
    int N=10;;
    int *a= (int*)malloc(N*sizeof(int));
    int *b= (int*)malloc(N*sizeof(int));
    int *c= (int*)malloc(N*sizeof(int));

    for(int i=0;i<N;i++){
        a[i]=1-i;
        b[i]=1+i;
    }

    add(a,b,c,N);

    for(int i=0;i<N;i++){
        printf("%d ",c[i]);
    }
    printf("\n");
    free(a);
    free(b);
    free(c);
    return 0;
}


2 2 2 2 2 2 2 2 2 2 



In [9]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include<stdio.h>
#include<stdlib.h>
#include<cuda.h>

// Device - GPU
// Host - CPU

__global__ void add(int *a, int *b, int *c, int N){
    int i= threadIdx.x+blockIdx.x*blockDim.x;
    if(i<N) c[i]=a[i]+b[i];
}

int main(int argc, char** argv){
    int N=10;;
    int *h_a= (int*)malloc(N*sizeof(int));
    int *h_b= (int*)malloc(N*sizeof(int));
    int *h_c= (int*)malloc(N*sizeof(int));

    for(int i=0;i<N;i++){
        h_a[i]=1-i;
        h_b[i]=1+i;
    }


    int *d_a, *d_b, *d_c;
    cudaMalloc((void**)&d_a, N*sizeof(int)); //needs pointer to the pointer for changing its value
    cudaMalloc((void**)&d_b, N*sizeof(int));
    cudaMalloc((void**)&d_c, N*sizeof(int));

    cudaMemcpy(d_a, h_a, N*sizeof(int), cudaMemcpyHostToDevice); //copy from host to device
    cudaMemcpy(d_b, h_b, N*sizeof(int), cudaMemcpyHostToDevice);

    int NthreadsPerBlock= 10;
    int NthreadBlocks = (N+NthreadsPerBlock-1)/NthreadsPerBlock;

    add<<<NthreadBlocks, NthreadsPerBlock>>>(d_a,d_b,d_c,N);

    cudaMemcpy(h_c, d_c, N*sizeof(int), cudaMemcpyDeviceToHost); //copy from device to host


    for(int i=0;i<N;i++){
        printf("%d ",h_c[i]);
    }
    printf("\n");

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);
    return 0;
}


2 2 2 2 2 2 2 2 2 2 



Array Reversal

In [16]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include<stdio.h>
#include<stdlib.h>
#include<cuda.h>
__global__ void reverse(int *a, int *b, int N){
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    if(i<N){
        b[N-i-1]=a[i];
    }
}
int main(){
    int N=10;
    int *h_a= (int*)malloc(N*sizeof(int));
    int *h_b= (int*)malloc(N*sizeof(int));
    int *d_a, *d_b;
    cudaMalloc(&d_a, N*sizeof(int));
    cudaMalloc(&d_b, N*sizeof(int));

    for(int i=0;i<N;i++){
        h_a[i]=i;
    }
    cudaMemcpy(d_a,h_a, N*sizeof(int),cudaMemcpyHostToDevice);

    int ntpb=5;
    int nb=(N+ntpb-1)/ntpb;
    reverse<<<nb,ntpb>>>(d_a,d_b,N);

    cudaMemcpy(h_b,d_b, N*sizeof(int), cudaMemcpyDeviceToHost);

    for(int i=0;i<N;i++){
        printf("%d",h_b[i]);
    }
    cudaFree(d_a);
    cudaFree(d_b);
    free(h_a);
    free(h_b);
    return 0;
}

9876543210


In place reverse

In [22]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include<stdio.h>
#include<stdlib.h>
#include<cuda.h>
__global__ void reverse(int *a, int N){
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    if(i<N/2){
        int temp1=a[i];
        a[i]=a[N-i-1];
        a[N-i-1]=temp1;
    }
}
int main(){
    int N=10;
    int *h_a= (int*)malloc(N*sizeof(int));
    int *d_a;
    cudaMalloc(&d_a, N*sizeof(int));

    for(int i=0;i<N;i++){
        h_a[i]=i;
    }
    cudaMemcpy(d_a,h_a, N*sizeof(int),cudaMemcpyHostToDevice);

    int ntpb=5;
    int nb=(N+ntpb-1)/ntpb;
    reverse<<<nb,ntpb>>>(d_a,N);

    cudaMemcpy(h_a,d_a, N*sizeof(int), cudaMemcpyDeviceToHost);

    for(int i=0;i<N;i++){
        printf("%d",h_a[i]);
    }
    cudaFree(d_a);
    free(h_a);
    return 0;
}

9876543210
